In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [3]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-4.1-nano"

In [5]:
###  1. 코드 인터프리터로 수학 방정식 풀이
math_prompt = "x^2 - 5x + 6 = 0 방정식의 해를 구해주세요."

math_response = client.responses.create(
    model=Model,
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto"}
        }
    ],
    instructions="당신은 수학 문제를 풀이하는 과외 선생님입니다. Python 코드로 계산한 후 해설도 포함하세요.",
    input=math_prompt,
)

print("\n 수학 문제 응답 결과:")
print(math_response.output_text)


 수학 문제 응답 결과:
이 방정식은 2차 방정식이므로, 근의 공식을 사용하거나 인수분해를 통해 해를 찾을 수 있습니다. 여기서는 인수분해 방법을 사용할 수 있습니다:

x^2 - 5x + 6 = (x - 2)(x - 3) = 0

따라서, 두 해는 각각 x = 2와 x = 3입니다.

이해를 돕기 위해 Python 코드를 사용해서 계산해보겠습니다. 

```python
import sympy as sp

x = sp.symbols('x')
equation = x**2 - 5*x + 6
solutions = sp.solve(equation, x)
solutions
```

이 코드를 실행해서 해를 구하면, x = 2와 x = 3이 출력됩니다.


In [9]:
### 2. 파일 검색 기반 질의응답 (삼성전자 사업보고서 활용)
# 벡터 스토어 생성
vector_store = client.vector_stores.create(name="삼성전자_사업보고서")

# PDF 파일 업로드
file_path = "재무제표/삼성전자사업보고서.pdf"  # 사전 존재 가정
file_stream = open(file_path, "rb")
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id, files=[file_stream]
)

print("\n 파일 업로드 상태:", file_batch.status)

# 질의 응답
query = "2025년 삼성전자의 반도체 부문 실적 요약을 알려주세요."
response = client.responses.create(
    model=Model,
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }]
)

print("\n 파일 기반 응답:")
print(response.output_text)


 파일 업로드 상태: completed

 파일 기반 응답:
2025년 삼성전자의 반도체 부문 실적 요약은 다음과 같습니다. 삼성전자는 2025년 1분기 기준, 총 매출이 79조 1,405억원으로 전년 동기 대비 10.0% 증가하였으며, 부문별 매출은 DX 부문이 51조 7,172억 원, DS 부문이 25조 1,310억 원, SDC가 5조 8,669억 원, Harman이 3조 4,194억 원입니다. 

반도체 부문의 경우, 메모리 제품인 DRAM과 NAND 플래시, 모바일 애플리케이션 프로세서(SoC) 등 핵심 제품의 생산과 판매가 지속되고 있으며, 1분기 메모리 생산 실적은 468,475,889개(1Gb 환산 기준)로 나타났습니다. 또한, 디스플레이 패널 부문도 성장세를 보이고 있으며, 스마트폰용 OLED 패널의 시장 점유율도 계속 확대되고 있습니다. 

매출과 함께, 반도체 부문의 영업이익은 1조 1,055억 원으로 보고되어, 수요 회복과 함께 수익성 향상에 기여하고 있음을 보여줍니다. 투자 현황에 있어서도 첨단 공정 증설 및 인프라 투자를 지속하며, 1분기 시설투자는 12조  bodies 훙원을 기록하였고, 글로벌 생산능력의 활용도를 높이고 있습니다.

이 내용은 삼성전자가 2025년 1분기까지 반도체 부문에서 이룬 성과와 투자 현황, 시장 동향 등을 포괄적으로 보여줍니다.
